# 01 - Ingesta ESIOS -> raw

Inventaría los indicadores disponibles en ESIOS y permite descargar, sin transformar, las respuestas originales de todos ellos para el periodo configurado. Los datos se guardan localmente en `data/raw/esios/`. La subida a S3 está desactivada por defecto y solo se activa explícitamente si se dispone de credenciales AWS.

**Importante:** descargar todos los indicadores puede generar muchas peticiones y archivos. Ejecuta primero el inventario y revisa la estimación. La descarga completa solo se inicia al cambiar `RUN_FULL_DOWNLOAD` a `True`. La API puede ofrecer datos vacíos para ciertos indicadores o periodos; esos JSON se conservan igualmente. No implica que los indicadores sean adecuados para el modelo.

In [8]:
import json
import os
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto (requirements.txt y notebooks/).")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("ESIOS_API_KEY")
BUCKET = os.getenv("S3_BUCKET")
UPLOAD_TO_S3 = False
BASE_URL = "https://api.esios.ree.es"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "esios"

if not API_KEY:
    raise RuntimeError(f"Falta ESIOS_API_KEY en {PROJECT_ROOT / '.env'}.")
if UPLOAD_TO_S3 and not BUCKET:
    raise RuntimeError("UPLOAD_TO_S3 está activado, pero falta S3_BUCKET en el archivo .env.")

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Destino local: {RAW_DIR}")
print(f"Subida S3: {'activada' if UPLOAD_TO_S3 else 'desactivada'}")

Raíz del proyecto: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer
Destino local: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\data\raw\esios
Subida S3: desactivada


In [9]:
# Periodo solicitado: año de datos del proyecto.
START_DATE = date(2026, 7, 1)
END_DATE = date(2026, 9, 30)

# IDs a descargar. None selecciona todo el catálogo; una lista limita la ingesta.
INDICATOR_IDS: list[int] | None = [541, 542]

# Controles para la descarga masiva.
CHUNK_DAYS = 30
PAUSE_SECONDS = 1
RETRIES = 4
FORCE = False
RUN_FULL_DOWNLOAD = True

if END_DATE < START_DATE:
    raise ValueError("END_DATE debe ser igual o posterior a START_DATE.")

print(f"Rango UTC: {START_DATE} a {END_DATE}")

Rango UTC: 2026-07-01 a 2026-09-30


In [10]:
HEADERS = {
    "Accept": "application/json; application/vnd.esios-api-v1+json",
    "Content-Type": "application/json",
    "x-api-key": API_KEY,
}


def get_json(url: str, params: dict | None = None) -> dict:
    for attempt in range(RETRIES):
        try:
            response = requests.get(url, headers=HEADERS, params=params, timeout=90)
            if response.status_code in (401, 403):
                raise PermissionError(
                    f"ESIOS rechazó la credencial o el acceso (HTTP {response.status_code}). "
                    "Revisa ESIOS_API_KEY y sus permisos."
                )
            if response.status_code == 429 or response.status_code >= 500:
                response.raise_for_status()
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict):
                raise ValueError("ESIOS devolvió una respuesta JSON que no es un objeto.")
            return payload
        except requests.RequestException as exc:
            if attempt == RETRIES - 1:
                raise RuntimeError(f"Falló la petición a ESIOS: {url}") from exc
            wait = 2 ** attempt
            print(f"Petición fallida; reintento {attempt + 1}/{RETRIES} en {wait}s: {exc}")
            time.sleep(wait)


def save_raw(payload: dict, path: Path, s3_key: str | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    if UPLOAD_TO_S3 and s3_key:
        import boto3

        boto3.client("s3").upload_file(str(path), BUCKET, s3_key)
        print(f"  subido a s3://{BUCKET}/{s3_key}")
    return path


def date_chunks(start: date, end: date, size: int = CHUNK_DAYS):
    current = start
    while current <= end:
        chunk_end = min(current + timedelta(days=size - 1), end)
        yield current, chunk_end
        current = chunk_end + timedelta(days=1)


def values_count(payload: dict) -> int:
    indicator = payload.get("indicator")
    if not isinstance(indicator, dict) or not isinstance(indicator.get("values"), list):
        raise ValueError("Respuesta de indicador no válida: se esperaba indicator.values como lista.")
    return len(indicator["values"])

## 1. Inventario del catálogo

Esta celda consulta y guarda la respuesta del catálogo sin modificarla. El número de indicadores y la estimación de llamadas dependen de lo que ESIOS publique al ejecutar el notebook.

In [11]:
catalog_payload = get_json(f"{BASE_URL}/indicators", params={"locale": "es"})
indicators = catalog_payload.get("indicators")
if not isinstance(indicators, list):
    raise ValueError("Respuesta del catálogo no válida: falta la lista 'indicators'.")

downloaded_at = datetime.now(timezone.utc)
catalog_name = f"catalogo_indicadores_{downloaded_at.strftime('%Y%m%dT%H%M%SZ')}.json"
catalog_path = RAW_DIR / "catalogo" / catalog_name
save_raw(catalog_payload, catalog_path, f"raw/esios/catalogo/{catalog_name}")

catalog_ids_all = sorted({
    int(item["id"])
    for item in indicators
    if isinstance(item, dict) and item.get("id") is not None
})
if INDICATOR_IDS is None:
    catalog_ids = catalog_ids_all
else:
    unknown_ids = sorted(set(INDICATOR_IDS) - set(catalog_ids_all))
    if unknown_ids:
        raise ValueError(f"IDs no encontrados en el catálogo ESIOS: {unknown_ids}")
    catalog_ids = sorted(set(INDICATOR_IDS))
n_chunks = sum(1 for _ in date_chunks(START_DATE, END_DATE))
estimated_requests = len(catalog_ids) * n_chunks

print(f"Indicadores disponibles en catálogo: {len(catalog_ids_all)}")
print(f"Indicadores seleccionados para ingesta: {len(catalog_ids)} {catalog_ids}")
print(f"Bloques por indicador: {n_chunks}")
print(f"Peticiones estimadas: {estimated_requests:,}")
print(f"Inventario original guardado en: {catalog_path}")

Indicadores en catálogo: 1982
Bloques por indicador: 4
Peticiones estimadas para todas las series: 7,928
Inventario original guardado en: c:\Users\zhen.xiang\OneDrive - Accenture\Documents\SAT\Energy\odb-challenge-energy-optimizer\data\raw\esios\catalogo\catalogo_indicadores_20261111T110325Z.json


## 2. Descarga de todas las series

Revisa la estimación anterior antes de continuar. Para lanzar la descarga, cambia `RUN_FULL_DOWNLOAD` a `True` en la celda de configuración y vuelve a ejecutar las celdas desde arriba. Los bloques guardados se omiten, por lo que se puede reanudar; usa `FORCE = True` para volver a descargarlos.

In [ ]:
if not RUN_FULL_DOWNLOAD:
    print("Descarga masiva desactivada. Revisa la estimación y cambia RUN_FULL_DOWNLOAD a True para iniciar.")
else:
    downloaded = 0
    skipped = 0
    empty = 0
    failures = []
    chunks = list(date_chunks(START_DATE, END_DATE))

    for indicator_id in catalog_ids:
        for chunk_start, chunk_end in chunks:
            filename = f"indicador_{indicator_id}_{chunk_start}_{chunk_end}.json"
            local_path = RAW_DIR / "indicadores" / str(indicator_id) / filename
            s3_key = f"raw/esios/indicadores/{indicator_id}/{filename}"

            if local_path.exists() and not FORCE:
                skipped += 1
                continue

            url = f"{BASE_URL}/indicators/{indicator_id}"
            params = {
                "start_date": f"{chunk_start.isoformat()}T00:00:00Z",
                "end_date": f"{chunk_end.isoformat()}T23:59:59Z",
                "locale": "es",
            }
            try:
                payload = get_json(url, params=params)
                n_values = values_count(payload)
                save_raw(payload, local_path, s3_key)
                downloaded += 1
                empty += n_values == 0
                print(f"Indicador {indicator_id} | {chunk_start}..{chunk_end} | {n_values} valores")
            except (RuntimeError, PermissionError, ValueError) as exc:
                failures.append((indicator_id, chunk_start, chunk_end, str(exc)))
                print(f"ERROR indicador {indicator_id} | {chunk_start}..{chunk_end}: {exc}")
            time.sleep(PAUSE_SECONDS)

    print(
        f"\nResumen: descargados={downloaded}, ya existentes={skipped}, "
        f"bloques vacíos={empty}, fallidos={len(failures)}"
    )
    if failures:
        failures_path = RAW_DIR / "fallos_ingesta.json"
        failures_path.parent.mkdir(parents=True, exist_ok=True)
        failures_path.write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError(f"Hay bloques fallidos; detalle guardado en {failures_path}.")

Indicador 1 | 2026-07-01..2026-07-30 | 2880 valores
Indicador 1 | 2026-07-31..2026-08-29 | 2880 valores
Indicador 1 | 2026-08-30..2026-09-28 | 2880 valores
Indicador 1 | 2026-09-29..2026-09-30 | 192 valores
Indicador 2 | 2026-07-01..2026-07-30 | 2880 valores
Indicador 2 | 2026-07-31..2026-08-29 | 2880 valores
Indicador 2 | 2026-08-30..2026-09-28 | 2880 valores
Indicador 2 | 2026-09-29..2026-09-30 | 192 valores
Indicador 3 | 2026-07-01..2026-07-30 | 1669 valores
Indicador 3 | 2026-07-31..2026-08-29 | 1765 valores
Indicador 3 | 2026-08-30..2026-09-28 | 1837 valores
Indicador 3 | 2026-09-29..2026-09-30 | 120 valores
Indicador 4 | 2026-07-01..2026-07-30 | 2880 valores
Indicador 4 | 2026-07-31..2026-08-29 | 2880 valores
Indicador 4 | 2026-08-30..2026-09-28 | 2880 valores
Indicador 4 | 2026-09-29..2026-09-30 | 192 valores
Indicador 5 | 2026-07-01..2026-07-30 | 0 valores
Indicador 5 | 2026-07-31..2026-08-29 | 0 valores
Indicador 5 | 2026-08-30..2026-09-28 | 0 valores
Indicador 5 | 2026-09-29.